# Model Review

Lightweight notebook for reviewing saved model outputs, signal quality, and cumulative returns across the ridge, tree, and MLP regression runs.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
pd.options.display.float_format = '{:,.4f}'.format

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
PRED_DIR = ROOT / 'outputs' / 'predictions'
METRICS_DIR = ROOT / 'outputs' / 'metrics'

models = ['ridge', 'tree', 'mlp']
display_names = {'ridge': 'Ridge', 'tree': 'Tree', 'mlp': 'MLP'}

predictions = {
    model: pd.read_csv(PRED_DIR / f'{model}_regression_predictions.csv', parse_dates=['date'])
    for model in models
}
portfolio_returns = {
    model: pd.read_csv(METRICS_DIR / f'{model}_regression_portfolio_returns.csv', parse_dates=['date'])
    for model in models
}
comparison = pd.read_csv(METRICS_DIR / 'model_comparison_regression.csv')
comparison

In [ ]:
signal_quality = []
for model, df in predictions.items():
    daily_ic = df.groupby('date').apply(lambda x: x['prediction'].corr(x['realized_return'], method='spearman'))
    signal_quality.append({
        'model': display_names[model],
        'mean_ic': daily_ic.mean(),
        'std_ic': daily_ic.std(ddof=0),
        'prediction_correlation': df['prediction'].corr(df['realized_return']),
    })

signal_quality = pd.DataFrame(signal_quality).sort_values('mean_ic', ascending=False).reset_index(drop=True)
signal_quality

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for model, df in portfolio_returns.items():
    ax.plot(df['date'], df['equity_curve'], linewidth=2, label=display_names[model])
ax.set_title('Cumulative Net Equity Curve by Model')
ax.set_xlabel('Date')
ax.set_ylabel('Equity Curve')
ax.legend(title='Model', frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, model in zip(axes, models):
    df = predictions[model]
    sample = df.sample(min(len(df), 1500), random_state=42)
    ax.scatter(sample['prediction'], sample['realized_return'], alpha=0.35, s=12)
    ax.set_title(f"{display_names[model]}: Prediction vs Realized Return")
    ax.set_xlabel('Predicted 5D Return')
axes[0].set_ylabel('Realized 5D Forward Return')
plt.tight_layout()
plt.show()

In [ ]:
summary_columns = [
    'model',
    'rmse',
    'mae',
    'correlation',
    'spearman_rank_correlation',
    'mean_ic',
    'annualized_return',
    'annualized_volatility',
    'sharpe_ratio',
    'max_drawdown',
    'turnover',
]

comparison.assign(model=comparison['model'].map(display_names))[summary_columns]